<a href="https://colab.research.google.com/github/hadasecohen/streaming-vae-anomaly-detection/blob/main/scripts/compile_test_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cython compile test (Colab / Linux)

**Purpose:** verify `src/` compiles cleanly with Cython on Colab's own Linux runtime and Python version, and that the resulting `.so` files import and run correctly with no `.py` source present. This is a feasibility/build check, not part of the normal demo — it clones the repo into its own directory and does not touch `demo_colab.ipynb` or any other notebook.

Colab's runtime is Linux/x86_64 with `gcc` preinstalled, so no compiler setup is needed (unlike a local Windows machine, which needs MSVC Build Tools installed separately).

### Step 1 — Clone the repo

This repo is currently private, so cloning needs a GitHub token — same flow as `demo_colab.ipynb`: add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) with a token that has read access to this repo, and enable **Notebook access** for it.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo_compile_test")

from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError(
        "The Colab secret GITHUB_TOKEN is unavailable. "
        "Open the key icon in the left sidebar, add the token, "
        "and enable Notebook access."
    )

if not REPO_DIR.exists():
    # Use an askpass helper so the token is not stored in the Git remote URL.
    askpass = Path("/content/git_askpass_compile_test.sh")
    askpass_body = (
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    )
    askpass.write_text(askpass_body)
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True,
            env=env,
        )
    finally:
        askpass.unlink(missing_ok=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

### Step 2 — Install Cython and confirm a C compiler is available

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cython"], check=True)

gcc_check = subprocess.run(["gcc", "--version"], capture_output=True, text=True)
print(gcc_check.stdout.splitlines()[0] if gcc_check.returncode == 0 else "gcc NOT found")

import Cython
print("Cython version:", Cython.__version__)
print("Python version:", sys.version)

### Step 3 — Run the compile script

This is the same `scripts/local/compile_src.py` used to test-compile locally — copies `src/` into `build/compiled_src/src`, Cythonizes every `.py` file except `__init__.py`, and builds `.so` extension modules with `annotation_typing=False` (this codebase's type hints aren't reliable contracts everywhere; see the script's own docstring).

In [ ]:
!python scripts/local/compile_src.py --out build/compiled_src 2>&1 | tail -60

### Step 4 — Confirm all expected `.so` files were produced

In [ ]:
import glob

so_files = sorted(glob.glob("build/compiled_src/src/**/*.so", recursive=True))
print(f"{len(so_files)} compiled .so files:")
for f in so_files:
    print(" ", f)

### Step 5 — Delete the `.py` source from the build output, keep only `.so`

Python prefers `.py` over `.so` on import, so the compiled modules would be silently ignored if the source were still sitting next to them. This step proves the compiled package is self-sufficient with no source present.

In [ ]:
import glob, os

removed = 0
for f in glob.glob("build/compiled_src/src/**/*.py", recursive=True):
    if os.path.basename(f) != "__init__.py":
        os.remove(f)
        removed += 1
print(f"Removed {removed} .py source files (kept __init__.py package markers)")

### Step 6 — Import the compiled-only package and run a real forward pass

In [ ]:
import sys
sys.path.insert(0, "build/compiled_src")

import torch
from src.VAE.mlp_VAE import MLP_VAE
from src.VAE.base_VAE import LossFunc
from src.Data_tools.dataset_loader import data_fn_builder
from src.env.streaming_env_optuna import StreamEnvOptuna
from src.env.streaming_env_standalone import StreamEnvStandalone

print("All modules imported from compiled .so with no .py source present")
for mod_name in ["src.VAE.mlp_VAE", "src.Data_tools.dataset_loader", "src.env.streaming_env_optuna"]:
    mod = sys.modules[mod_name]
    print(f"  {mod_name:35s} -> {mod.__file__}")

loss_func = LossFunc(loss_fn_type="elbo", recon_loss_kind="mse")
model = MLP_VAE(
    input_dim=10, latent_dim=4, loss_func=loss_func, pooling="mean",
    prior_variance=1.0,
    enc_dec_hidden_dims=[32, 32], enc_dec_activation_funcs=["relu", "relu"],
    enc_dec_dropouts=[0.0, 0.0],
    clamp_logvar=True, clamp_bounds=[-4.0, 4.0],
)
out = model(torch.randn(2, 1, 10))
print("\nForward pass OK. Param count:", sum(p.numel() for p in model.parameters()))
print("\nSUCCESS: src/ compiles and runs correctly on Colab's Linux/Python target.")

### Step 7 — Swap in the compiled src/ and run demo.py for real

Steps 4-6 proved the compiled package imports and runs a synthetic forward pass with no `.py` source present, but that was still importing from `build/compiled_src/`, not the repo's actual `src/` that `demo.py` and every notebook actually use.

This step backs up the real `src/` (so it can be restored), replaces it with the compiled-only version, removes the leftover `.c` intermediate files Cython leaves behind, and runs `demo.py` — the same thing a user would run — against the compiled package. Everything here stays inside the disposable `/content/repo_compile_test` clone; your local repo's `src/` is never touched.

In [ ]:
import shutil
import glob
import os
from pathlib import Path

REPO_DIR = Path("/content/repo_compile_test")
real_src = REPO_DIR / "src"
compiled_src = REPO_DIR / "build" / "compiled_src" / "src"
backup_src = REPO_DIR / "src_original_backup"

assert compiled_src.exists(), "Run Steps 3-5 first to produce and clean build/compiled_src/src"

# Remove leftover .c intermediate files (compiled .so files stay).
c_files = glob.glob(str(compiled_src / "**" / "*.c"), recursive=True)
for f in c_files:
    os.remove(f)
print(f"Removed {len(c_files)} leftover .c intermediate files")

# Back up the real src/, then swap in the compiled one.
if backup_src.exists():
    shutil.rmtree(backup_src)
shutil.move(str(real_src), str(backup_src))
shutil.copytree(str(compiled_src), str(real_src))
print(f"Swapped in compiled src/ (backup at {backup_src})")

so_count = len(glob.glob(str(real_src / "**" / "*.so"), recursive=True))
py_count = len([
    f for f in glob.glob(str(real_src / "**" / "*.py"), recursive=True)
    if os.path.basename(f) != "__init__.py"
])
print(f"src/ now has {so_count} .so files and {py_count} non-init .py files (should be 0)")

In [ ]:
# Run the real demo.py against the compiled src/. Point mode is fastest
# (CPU-only, no batching) — good for a quick end-to-end check.
!python demo.py --anom point --arch mlp

### Step 8 — Restore the original src/ (optional cleanup)

In [ ]:
# Restore the original .py src/ afterward, in case you want to re-run
# earlier cells or inspect the uncompiled source again in this session.
import shutil
from pathlib import Path

REPO_DIR = Path("/content/repo_compile_test")
real_src = REPO_DIR / "src"
backup_src = REPO_DIR / "src_original_backup"

if backup_src.exists():
    shutil.rmtree(real_src)
    shutil.move(str(backup_src), str(real_src))
    print("Restored original src/ from backup")
else:
    print("No backup found (already restored, or Step 7 wasn't run)")